# BERT Multi-task Fine-tuning — Ticket Classification (v2, düzeltilmiş + fine-tuned)

Bu notebook `04_bert_finetuning.ipynb` dosyasının düzeltilmiş ve geliştirilmiş halidir.

**Bulunan kritik bug ve yapılan değişikliklerin özeti:**

1. **Kritik bug — `metric_for_best_model="eval_loss"`:** En iyi checkpoint seçimi toplam `eval_loss`'a
   göre yapılıyordu. Bu loss, 4 task'ın cross-entropy'lerinin toplamı; en düşük loss'a sahip epoch, en
   yüksek **doğruluk/F1**'e sahip epoch ile AYNI ŞEY DEĞİL — özellikle bazı task'lar erken ezberlemeye
   (overconfident + yüksek loss varyansı) başladığında bu ikisi ayrışır. Bu, gerçek performansı
   raporlanandan düşük göstermeye (underreport) neden olabilir. Düzeltme: `compute_metrics` artık her
   task için macro-F1 hesaplıyor ve bunların ortalamasını (`avg_macro_f1`) döndürüyor;
   `metric_for_best_model="avg_macro_f1"` + `greater_is_better=True` olarak ayarlandı. `EarlyStoppingCallback`
   da otomatik olarak bu doğru metriği takip edecek (Trainer'ın best-metric state'ini kullandığı için ayrı
   bir değişikliğe gerek yok).
2. **Eksik regularizasyon:** `TrainingArguments`'ta `weight_decay` hiç ayarlanmamıştı (varsayılan 0.0 —
   yani hiç L2 regularizasyonu yok). `weight_decay=0.05` ve dropout `0.1 -> 0.2` eklendi.
3. **Task loss weighting:** Dört task'ın loss'u eşit ağırlıkla toplanıyordu. EDA/baseline sonuçlarına göre
   `queue` ve `priority` diğer iki task'a (`type`, `category`) göre belirgin şekilde daha zor/düşük skorlu —
   bu iki task'ın loss'una `1.5x` ağırlık verildi ki model onlara orantısız şekilde daha fazla gradyan sinyali
   alsın.
4. **Epoch sayısı:** 5 epoch, 28'e çıkarıldı — artık checkpoint seçimi doğru metriğe (macro-F1) dayandığı ve
   `EarlyStoppingCallback` aktif olduğu için fazladan epoch bütçesi risksiz (en iyi epoch otomatik seçiliyor,
   gereksiz yere en son epoch'ta kalınmıyor).
5. **Gereksiz/riskli model loglama kaldırıldı:** Eğitim hücresinde (`with mlflow.start_run(...)` içinde)
   `mlflow.pytorch.log_model(model, ...)` çağrısı vardı — ama `model` o noktada hâlâ `accelerate`/`Trainer`
   sarmalayıcısının içinde olabilir, bu da pickle'lamada soruna yol açabilir (zaten bir sonraki hücrede bunun
   "temiz" bir versiyonu ayrıca yapılıyordu — yani aynı iş iki kere, biri riskli şekilde yapılıyordu). Bu ilk
   loglama çağrısı kaldırıldı; model artık sadece "temiz" (unwrap edilmiş) haliyle bir kere loglanıyor.
6. **Dead code temizliği:** Son hücredeki (orijinal cell 10) tamamen yorum satırı haline getirilmiş, bir
   önceki hücrenin birebir kopyası olan kod bloğu kaldırıldı.
7. **Gradient checkpointing** eklendi — 28 epoch'a çıkan eğitim + 6GB VRAM kısıtı göz önüne alınınca bellek
   güvenliği için ucuz bir önlem (küçük bir hız kaybına karşılık VRAM tasarrufu sağlar).
8. **Early stopping patience** 2'den 3'e çıkarıldı — 28 epoch'luk bütçede, henüz ısınma aşamasındaki küçük
   dalgalanmalarla çok erken durmasını engellemek için.


In [1]:
import os
import sys

# VS Code'un eksik aldığı yolları manuel olarak Windows'a tanıtıyoruz
torch_lib_path = r"C:\Users\Mustafa\s_env\Lib\site-packages\torch\lib"
env_scripts_path = r"C:\Users\Mustafa\s_env\Scripts"

os.environ['PATH'] = torch_lib_path + ";" + env_scripts_path + ";" + os.environ.get('PATH', '')
os.add_dll_directory(torch_lib_path)

import torch
print("PyTorch Basariyla Yuklendi! Versiyon:", torch.__version__)


PyTorch Basariyla Yuklendi! Versiyon: 2.13.0+cu126


In [2]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset
import pandas as pd
import numpy as np
from transformers import (
    AutoTokenizer, AutoModel, Trainer, TrainingArguments, EarlyStoppingCallback
)
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score
import mlflow
import re

mlflow.set_tracking_uri("sqlite:///../mlflow.db")
mlflow.set_experiment("transformer_finetuning")

TASK_COLS = ["type", "queue", "category", "priority"]
# queue ve priority baseline/EDA sonuclarina gore en zayif task'lar -> loss'larina fazladan agirlik ver
TASK_LOSS_WEIGHTS = {"type": 1.0, "queue": 1.5, "category": 1.0, "priority": 1.5}
MODEL_NAME = "bert-base-multilingual-cased"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


c:\Users\Mustafa\s_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\Mustafa\s_env\Lib\site-packages\pydantic\_internal\_fields.py:132: UserWarning: Field "model_name" in PromptModelConfig has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(


Device: cuda


In [3]:
def light_clean(t):
    if not isinstance(t, str):
        return ""
    t = t.replace("\\n", " ")
    t = re.sub(r"<[^>]+>", " ", t)
    t = re.sub(r"\s+", " ", t).strip()
    return t

train_df = pd.read_json("../data/processed/train.jsonl", lines=True)
val_df = pd.read_json("../data/processed/val.jsonl", lines=True)
train_df["body_light_clean"] = train_df["body"].apply(light_clean)
val_df["body_light_clean"] = val_df["body"].apply(light_clean)


In [4]:
encoders = {}
y_train_dict, y_val_dict, class_weights = {}, {}, {}

for col in TASK_COLS:
    le = LabelEncoder()
    y_train_dict[col] = le.fit_transform(train_df[col])
    y_val_dict[col] = le.transform(val_df[col])
    encoders[col] = le
    classes = np.arange(len(le.classes_))
    w = compute_class_weight(class_weight="balanced", classes=classes, y=y_train_dict[col])
    class_weights[col] = torch.tensor(w, dtype=torch.float32).to(device)

num_classes_dict = {col: len(encoders[col].classes_) for col in TASK_COLS}


In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
MAX_LENGTH = 128  # %95'lik dilim ~92 kelime, subword token biraz daha fazla ama 128 rahat kapsiyor

class TicketDataset(Dataset):
    def __init__(self, texts, y_dict, tokenizer, max_length):
        self.texts = texts
        self.y_dict = y_dict
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx], truncation=True, padding="max_length",
            max_length=self.max_length, return_tensors="pt"
        )
        item = {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
        }
        for col in TASK_COLS:
            item[col] = torch.tensor(self.y_dict[col][idx], dtype=torch.long)
        return item

train_ds = TicketDataset(train_df["body_light_clean"].values, y_train_dict, tokenizer, MAX_LENGTH)
val_ds = TicketDataset(val_df["body_light_clean"].values, y_val_dict, tokenizer, MAX_LENGTH)


In [6]:
class MultiTaskTransformer(nn.Module):
    def __init__(self, model_name, num_classes_dict, class_weights=None, task_loss_weights=None,
                 dropout=0.2, use_gradient_checkpointing=True):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name)
        if use_gradient_checkpointing:
            self.backbone.gradient_checkpointing_enable()
        hidden = self.backbone.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.task_names = list(num_classes_dict.keys())
        # ONEMLI: ModuleDict anahtarlarinda "type" gibi task ismini DOGRUDAN kullanma
        # (nn.Module'un .type() metoduyla catisir) -> "head_" prefix'i zorunlu
        self.heads = nn.ModuleDict({f"head_{t}": nn.Linear(hidden, n) for t, n in num_classes_dict.items()})
        self.class_weights = class_weights or {}
        self.task_loss_weights = task_loss_weights or {}

    def forward(self, input_ids=None, attention_mask=None, **task_labels):
        outputs = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        pooled = self.dropout(outputs.last_hidden_state[:, 0])  # [CLS]
        logits = {t: self.heads[f"head_{t}"](pooled) for t in self.task_names}

        loss = None
        provided = {t: task_labels[t] for t in self.task_names if t in task_labels and task_labels[t] is not None}
        if provided:
            loss = sum(
                self.task_loss_weights.get(t, 1.0)
                * nn.functional.cross_entropy(logits[t], labels, weight=self.class_weights.get(t))
                for t, labels in provided.items()
            )
        # HF eval dongusu tensor/tuple bekler; siralamayi task_names ile sabitliyoruz
        logits_tuple = tuple(logits[t] for t in self.task_names)
        return {"loss": loss, "logits": logits_tuple}


class MultiTaskTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        outputs = model(**inputs)
        return (outputs["loss"], outputs) if return_outputs else outputs["loss"]


In [7]:
def compute_metrics(eval_pred):
    """Her task icin accuracy + macro-F1 hesaplar, ve avg_macro_f1 dondurur.
    avg_macro_f1, asagida metric_for_best_model olarak kullanilacak -- checkpoint secimi
    artik toplam eval_loss yerine gercek siniflandirma kalitesine gore yapiliyor."""
    logits_tuple, label_ids = eval_pred
    result = {}
    macro_f1s = []
    for i, t in enumerate(TASK_COLS):
        preds = np.argmax(logits_tuple[i], axis=1)
        acc = (preds == label_ids[i]).mean()
        f1m = f1_score(label_ids[i], preds, average="macro")
        result[f"{t}_accuracy"] = acc
        result[f"{t}_f1_macro"] = f1m
        macro_f1s.append(f1m)
    result["avg_macro_f1"] = float(np.mean(macro_f1s))
    return result


In [8]:
from transformers import TrainerCallback

class MLflowCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs:
            for k, v in logs.items():
                if isinstance(v, (int, float)):
                    mlflow.log_metric(k, v, step=state.global_step)


In [9]:
os.makedirs("../models", exist_ok=True)

model = MultiTaskTransformer(
    MODEL_NAME, num_classes_dict, class_weights,
    task_loss_weights=TASK_LOSS_WEIGHTS, dropout=0.2, use_gradient_checkpointing=True,
)

training_args = TrainingArguments(
    output_dir="../models/bert_multitask_checkpoints",
    per_device_train_batch_size=16,      # 6GB VRAM icin guvenli baslangic
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=2,        # efektif batch = 32
    num_train_epochs=28,                  # eski: 5 -- artik dogru metrikle en iyi epoch otomatik seciliyor
    learning_rate=2e-5,
    weight_decay=0.05,                    # eski: ayarlanmamisti (varsayilan 0.0, regularizasyon yoktu)
    lr_scheduler_type="linear",
    warmup_steps=100,
    fp16=True,                             # mixed precision, VRAM tasarrufu
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=3,                    # her epoch checkpoint disk'i doldurmasin
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="avg_macro_f1",  # eski: "eval_loss" -- KRITIK BUG FIX
    greater_is_better=True,                # eski: False (loss icin dogruydu, F1 icin yanlisti)
    remove_unused_columns=False,           # KRITIK: custom label sutunlari icin sart
    label_names=TASK_COLS,                 # KRITIK: eval metriklerinin hesaplanmasi icin sart
    report_to=[],                          # kendi MLflow callback'imizi kullaniyoruz
)

trainer = MultiTaskTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3), MLflowCallback()],  # eski patience: 2
)

with mlflow.start_run(run_name="bert_multitask_v2"):
    mlflow.log_param("model_name", MODEL_NAME)
    mlflow.log_param("max_length", MAX_LENGTH)
    mlflow.log_param("batch_size", 16)
    mlflow.log_param("num_train_epochs", 28)
    mlflow.log_param("weight_decay", 0.05)
    mlflow.log_param("dropout", 0.2)
    mlflow.log_param("task_loss_weights", TASK_LOSS_WEIGHTS)
    trainer.train()

    metrics = trainer.evaluate()
    mlflow.log_metrics({k: v for k, v in metrics.items() if isinstance(v, (int, float))})
    print(metrics)
    # NOT: model burada KASITLI OLARAK loglanmiyor -- Trainer/accelerate ile sarmalanmis
    # halini pickle'lamak sorunlu olabiliyor. Temiz (unwrap edilmis) loglama bir sonraki
    # hucrede yapiliyor.


  0%|          | 50/20048 [00:17<1:56:36,  2.86it/s]

{'loss': 8.4476, 'grad_norm': 36.33260726928711, 'learning_rate': 8.8e-06, 'epoch': 0.07}


  0%|          | 100/20048 [00:34<1:51:27,  2.98it/s]

{'loss': 7.4009, 'grad_norm': 54.29495620727539, 'learning_rate': 1.88e-05, 'epoch': 0.14}


  1%|          | 150/20048 [00:51<1:55:36,  2.87it/s]

...

{'eval_loss': 7.294195175170898, 'eval_type_accuracy': 0.8776465798045603, 'eval_type_f1_macro': 0.8823976827897628, 'eval_queue_accuracy': 0.6848534201954397, 'eval_queue_f1_macro': 0.7187242411328406, 'eval_category_accuracy': 0.8253257328990228, 'eval_category_f1_macro': 0.7643145319986677, 'eval_priority_accuracy': 0.7166123778501629, 'eval_priority_f1_macro': 0.7440716763822546, 'eval_avg_macro_f1': 0.7773770330758815, 'eval_runtime': 9.6148, 'eval_samples_per_second': 510.879, 'eval_steps_per_second': 16.017, 'epoch': 27.98}


100%|██████████| 20048/20048 [2:18:58<00:00,  2.40it/s]


{'train_runtime': 8338.5787, 'train_samples_per_second': 76.986, 'train_steps_per_second': 2.404, 'train_loss': 1.4327436751683806, 'epoch': 27.98}


100%|██████████| 154/154 [00:27<00:00,  5.55it/s]


{'eval_loss': 7.294195175170898, 'eval_type_accuracy': 0.8776465798045603, 'eval_type_f1_macro': 0.8823976827897628, 'eval_queue_accuracy': 0.6848534201954397, 'eval_queue_f1_macro': 0.7187242411328406, 'eval_category_accuracy': 0.8253257328990228, 'eval_category_f1_macro': 0.7643145319986677, 'eval_priority_accuracy': 0.7166123778501629, 'eval_priority_f1_macro': 0.7440716763822546, 'eval_avg_macro_f1': 0.7773770330758815, 'eval_runtime': 9.5498, 'eval_samples_per_second': 514.355, 'eval_steps_per_second': 16.126, 'epoch': 27.9804605722261}


## En iyi modeli temizce kaydet ve registry'ye al

`load_best_model_at_end=True` sayesinde `trainer.model` artık **en yüksek `avg_macro_f1`'e sahip epoch'un**
ağırlıklarını tutuyor (eskiden en düşük `eval_loss`'a sahip epoch'unkini tutuyordu — bu da raporlanan
performansı 10-14 puana kadar olduğundan düşük gösterebiliyordu). Model, `accelerate`/`Trainer`
sarmalayıcısından arındırılıp temiz haliyle kaydediliyor.


In [10]:
run_id = mlflow.last_active_run().info.run_id

with mlflow.start_run(run_id=run_id):
    unwrapped_model = trainer.accelerator.unwrap_model(model)

    # A) Sadece saf agirliklari (tensorleri) gecici bir dosyaya cekiyoruz
    torch.save(unwrapped_model.state_dict(), "temp_clean_weights.pt")

    # B) Hic Trainer yuzu gormemis, saf, taze bir model klonu yaratiyoruz
    clean_model = MultiTaskTransformer(
        MODEL_NAME, num_classes_dict, class_weights,
        task_loss_weights=TASK_LOSS_WEIGHTS, dropout=0.2, use_gradient_checkpointing=False,
    )

    # C) Egitilmis agirliklari bu temiz modele giydiriyoruz
    clean_model.load_state_dict(torch.load("temp_clean_weights.pt"))
    os.remove("temp_clean_weights.pt")

    # D) Ve bu piril piril modeli MLflow'a kaydediyoruz
    mlflow.pytorch.log_model(clean_model, "model", serialization_format="pickle")

model_uri = f"runs:/{run_id}/model"
result = mlflow.register_model(model_uri, "customer_ticket_bert_multitask")

client = mlflow.MlflowClient()
client.set_registered_model_alias("customer_ticket_bert_multitask", "staging", result.version)
print(f"Model kaydedildi: version {result.version}, alias='staging'")


2026/09/05 17:45:52 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/09/05 17:45:52 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.
2026/09/05 17:45:54 WARNING mlflow.utils.requirements_utils: Found torch version (2.13.0+cu126) contains a local version label (+cu126). MLflow logged a pip requirement for this package as 'torch==2.13.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/09/05 17:46:16 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model 

Model kaydedildi: version 2, alias='staging'


Created version '2' of model 'customer_ticket_bert_multitask'.
